In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import roc_auc_score

from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb


In [ ]:
# Loading the train dataset
train_dataset = pd.read_csv('data/train_dataset.csv')

In [ ]:
# Defining the function that will use the model to complete the missing values
# Using the model to complete the test dataset

def complete_dataset(model_name, model):

    test_dataset = pd.read_csv('data/test_dataset.csv')

    test_pred = model.predict(test_dataset)
    test_dataset['Transition'] = test_pred
    test_dataset.head()

    # Dropping all columns but the Transition column
    test_dataset.drop(test_dataset.columns.difference(['Transition']), axis=1, inplace=True)

    # Creating a RowId column to store the index, starting from 1
    test_dataset['RowId'] = np.arange(1, test_dataset.shape[0] + 1)

    # Placing the RowId column in the first position
    cols = test_dataset.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    test_dataset = test_dataset[cols]

    # Transforming the Transition column back to its original values
    replace_map = {'Transition': {0: 'CN-CN', 1: 'AD-AD', 2: 'CN-MCI', 3: 'MCI-AD', 4: 'MCI-MCI'}}
    test_dataset.replace(replace_map, inplace=True)
    test_dataset.head()

    # Saving the test dataset to a csv file
    test_dataset.to_csv('test_predictions_' + model_name + '.csv', index=False)

In [ ]:
# Define features and target
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

## Random Forest

In [ ]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import numpy as np

# Remove the 'oversampled' column from the dataset
X.drop(columns=['oversampled'], inplace=True)

# Define StratifiedKFold for non-oversampled data
stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)

# Define a baseline Random Forest Classifier
rfc = RandomForestClassifier(random_state=123)

# Define the hyperparameter distributions
param_dist = {'n_estimators': [50, 100, 200],
              'max_depth': [2, 6, 12, 20, None]}

# Split non-oversampled data
X_non_oversampled = X[train_dataset['oversampled'] == 'NO']
y_non_oversampled = y[train_dataset['oversampled'] == 'NO']

# Perform hyperparameter tuning using non-oversampled data only
random_search = RandomizedSearchCV(rfc, param_distributions=param_dist, n_iter=10, cv=stratified_kfold, scoring='f1_macro', random_state=123)
random_search.fit(X_non_oversampled, y_non_oversampled)

# Get the best model
best_rfc = random_search.best_estimator_

# Train the final model on the full dataset (original + oversampled) with the best hyperparameters
X_full = X
y_full = y
best_rfc.fit(X_full, y_full)

# Validate the model using cross-validation on non-oversampled data only
rfc_pred = cross_val_predict(best_rfc, X_non_oversampled, y_non_oversampled, cv=stratified_kfold)

# Calculate and print the Macro F1 Score
print('Random Forest Classifier')
print('Macro F1 Score:', f1_score(y_non_oversampled, rfc_pred, average='macro'))


In [ ]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_non_oversampled, rfc_pred))
# Plotting the confusion matrix with a caption
fig, ax = plt.subplots()
cax = ax.matshow(confusion_matrix(y_non_oversampled, rfc_pred), cmap=plt.cm.Blues)
fig.colorbar(cax)

# Adding labels
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix', pad=20)

# Adding caption
caption = "Confusion matrix for Random Forest Classifier"
plt.figtext(0.5, -0.1, caption, wrap=True, horizontalalignment='center', fontsize=12)

plt.show()

In [ ]:
# Printing the classification report

#print("Classification report")
#print(classification_report(y_val, rfc_pred))

In [ ]:
# Completing the test dataset
complete_dataset('random_forest', best_rfc)

## XGBoost

In [ ]:

# Define StratifiedKFold to maintain class balance
stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)

# Define the baseline model
xgbc = xgb.XGBClassifier(objective='multi:softprob', random_state=123, num_class=5)


# Uncomment this if already found the best hyperparameters
# Define the model with the best hyperparameters
best_xgbc = xgb.XGBClassifier(
    objective='multi:softprob', 
    random_state=123, 
    num_class=5, 
    n_estimators=200, 
    max_depth=3, 
    learning_rate=0.05
)
best_xgbc.fit(X_full, y_full) 

# Perform cross-validation with stratified folds
xgbc_pred = cross_val_predict(best_xgbc, X_non_oversampled, y_non_oversampled, cv=stratified_kfold)

# Calculate and display the F1 Score
print('XGBoost Classifier with StratifiedKFold')
print('Macro F1 Score:', f1_score(y_non_oversampled, xgbc_pred, average='macro'))

In [ ]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_non_oversampled, xgbc_pred))
# Plotting the confusion matrix with a caption
fig, ax = plt.subplots()
cax = ax.matshow(confusion_matrix(y_non_oversampled, xgbc_pred), cmap=plt.cm.Blues)
fig.colorbar(cax)

# Adding labels
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix', pad=20)

# Adding caption
caption = "Confusion matrix for XGBoost Classifier"
plt.figtext(0.5, -0.1, caption, wrap=True, horizontalalignment='center', fontsize=12)

plt.show()

In [ ]:
# Printing the classification report
#print("Classification report")
#print(classification_report(y_val, xgbc_pred))

In [ ]:
# Completing the test dataset
complete_dataset('xgboost', best_xgbc)